<!-- cabecera-entorno -->
## Antes de empezar

**Clase 5 · EDA: bivariado y multivariado** — Bloque 3 · Reto. Este cuaderno lo recorre **usted
solo**, leyendo: cada tarea trae la explicación y los comandos que necesita. El profesor circula por
el salón resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/saber_pro.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 5 · Reto — Análisis bivariado de los resultados Saber Pro

**Nombre:**

**Fecha:**

**Dataset:** `../datos/saber_pro.csv` — resultados Saber Pro (ICFES), vía datos.gov.co.
**Consigna completa:** `reto.md`

25.362 filas x 48 columnas. Una fila = un estudiante que presentó el examen entre 2018 y 2022.

### Las columnas que va a usar

| Columna | Qué mide | Tipo |
|---------|----------|------|
| `puntaje_global` | Puntaje total. **Es un compuesto de los otros cinco** | numérica |
| `punt_comp_ciud` | Competencias ciudadanas | numérica |
| `punt_comu_escr` | Comunicación escrita | numérica |
| `punt_ingles` | Inglés | numérica |
| `punt_lect_crit` | Lectura crítica | numérica |
| `punt_razo_cuant` | Razonamiento cuantitativo | numérica |
| `estrato` | Estrato 1 a 6, más `ND/NE` | categórica |
| `tipo_col` | Oficial, Privado, Otros, Sin datos | categórica |
| `sexo` | Hombres, Mujeres | categórica |
| `areac_snies` | Área de conocimiento del programa | categórica |

### Tres advertencias antes de escribir la primera línea

1. **Los faltantes están disfrazados de dato.** Las columnas de puntaje usan **-89** como código de
   dato faltante. No es un puntaje negativo: es un centinela. El archivo no tiene celdas vacías,
   tiene celdas mentirosas, y eso es peor. La celda de limpieza viene escrita; entenderla es parte
   del trabajo.
2. **`puntaje_global` no cuenta como hallazgo.** Se calcula a partir de los otros cinco puntajes, así
   que correlaciona alto con todos por construcción. Cuando le pidan las correlaciones más fuertes,
   va excluido.
3. **Hay 48 columnas y varias son numéricas sin ser cantidades.** `cod_dep_nac`, `snies_progra`,
   `lat_ciu_nac` son códigos y coordenadas. Correlacionarlos no significa nada.

## Cómo se recorre este cuaderno

Usted trabaja solo. Nadie va a dictar los pasos desde el tablero, así que cada tarea trae todo lo
que necesita para resolverse leyendo:

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, escrito en español |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones exactas que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué columna, qué lista, qué orden. Ahí no hay respuesta escrita |
| **La celda de código** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('TN', ...)` le dice si el resultado es el correcto, sin mostrárselo |

**Por qué esto sigue siendo un reto y no una copia.** En el demo trabajó sobre el consumo de agua de
Caldas. Aquí hay 25.362 estudiantes y 48 columnas que no vio nunca: usted elige la columna, elige la
lista de variables, decide qué gráfico corresponde a cada par de tipos y **dice qué significa el
número que sale**. La técnica se guía; el criterio no, y el criterio es lo que se evalúa. Las celdas
`comprobar(...)` comparan una huella digital de su resultado con la esperada: nunca revelan la
respuesta, y escribir cualquier cosa hasta que pasen es engañarse en el propio entregable.

**Las once tareas (T1 a T11)** están repartidas en tres partes. Las de gráfico se comprueban distinto:
no hay una única respuesta correcta para un gráfico, así que lo que se revisa es que esté dibujado,
titulado y con los dos ejes etiquetados, que es exactamente lo que pide la rúbrica.

**La parte C es la que más pesa** y es la única donde no se le da el orden de los comandos.

---

## Paso 0 · Preparación

**El concepto.** Cuatro librerías, y cada una hace una cosa distinta: **pandas** manipula la tabla,
**numpy** hace la aritmética por debajo, **matplotlib** dibuja, y **seaborn** dibuja gráficos
estadísticos (heatmap, boxplot, regplot) en una línea en vez de veinte.

Las dos primeras celdas ya están escritas. Ejecútelas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('pandas', pd.__version__, '| seaborn', sns.__version__)

In [ ]:
# Este cuaderno vive en clase05/reto/, y el CSV una carpeta más arriba, en data/
df = pd.read_csv('../datos/saber_pro.csv')

print(f'Filas y columnas: {df.shape}')
df.head()

### El verificador

La celda de abajo define `comprobar(...)` y `comprobar_grafico(...)`. Ejecútela una vez y siga
adelante: es andamiaje del curso, no materia de la clase.

In [ ]:
# Verificador de las once tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

_RESULTADOS = {}

_CLAVES = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10", "T11"]

_PISTAS = {
    "T1": "La matriz se pide con .corr() sobre la seleccion de columnas, no sobre el DataFrame completo. Son las SEIS columnas de columnas_puntaje, y sobre df_limpio, no sobre df.",
    "T2": "sns.heatmap(...) devuelve el eje: guardelo en una variable (eje = sns.heatmap(...)). Y la escala tiene que ir de -1 a 1: center=0, vmin=-1, vmax=1. Falta el titulo si el verificador lo dice.",
    "T3": "La matriz de este punto es la de columnas_modulo (cinco columnas, sin puntaje_global). Si le salio un par con puntaje_global adentro, uso la lista equivocada.",
    "T4": "Guarde el eje en una variable. plt.scatter(...) no devuelve el eje: use eje = plt.gca() despues de dibujar, o fig, eje = plt.subplots() antes. Titulo y las dos etiquetas.",
    "T5": "eje = sns.boxplot(x=..., y=..., data=..., order=orden_estratos). La categorica va en x y la numerica en y. Titulo y etiquetas en los dos ejes.",
    "T6": "Igual que la anterior pero con la columna del tipo de colegio. Aqui no hay orden logico que imponer, asi que puede omitir order.",
    "T7": "Es un groupby sobre la columna categorica, la columna numerica entre corchetes, y .agg(['mean', 'count']) al final. El conteo no es opcional: sin el, una media no se interpreta.",
    "T8": "El mismo groupby de la tarea anterior, cambiando la columna por la que agrupa. Deje las cuatro categorias tal como salen: filtrarlas cambia el resultado.",
    "T9": "pd.pivot_table con index, columns, values y aggfunc='mean'. Sobre df_limpio y sobre la columna de puntaje total. No filtre filas ni columnas todavia: la tabla completa.",
    "T10": "La misma llamada de la tarea anterior, cambiando unicamente aggfunc. 'size' cuenta filas por celda del cruce.",
    "T11": "Parta de la tabla de medias, quedese con las filas de orden_estratos y con las dos columnas que se comparan, y agregue una tercera columna con la resta. El orden de la resta importa: la ventaja se mide del privado hacia el oficial."
}

_ESPERADO = {
    "T1": "e53863a161",
    "T3": "8b7f7ad52a",
    "T7": "89f7129afb",
    "T8": "61a3e0037c",
    "T9": "ae82852930",
    "T10": "9211f7dbe9",
    "T11": "19d1debc7b"
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


def _redondear(valor, decimales):
    if decimales is None or valor is None:
        return valor
    if isinstance(valor, (pd.DataFrame, pd.Series)):
        return valor.round(decimales)
    try:
        return round(float(valor), decimales)
    except (TypeError, ValueError):
        return valor


def comprobar(clave, valor, decimales=None):
    """Dice si el resultado es el correcto, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        return
    valor = _redondear(valor, decimales)
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos.")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    if _huella(valor) == _ESPERADO.get(clave):
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def comprobar_grafico(clave, eje, con_ejes=True, escala_simetrica=False):
    """Revisa que el grafico exista y cumpla los requisitos que se califican.

    No hay una unica respuesta correcta para un grafico: lo que se comprueba es
    que este dibujado, titulado y etiquetado, que es lo que pide la rubrica.
    """
    _RESULTADOS[clave] = False
    if eje is None:
        print(f"[{clave}] Sin resolver todavia: la variable del eje sigue valiendo None.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    if not hasattr(eje, "get_title"):
        print(f"[{clave}] Eso no es un eje de matplotlib, es un {type(eje).__name__}.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    faltas = []
    if not eje.get_title().strip():
        faltas.append("falta el titulo: plt.title('...') o eje.set_title('...')")
    if con_ejes:
        if not eje.get_xlabel().strip():
            faltas.append("falta la etiqueta del eje x: plt.xlabel('...')")
        if not eje.get_ylabel().strip():
            faltas.append("falta la etiqueta del eje y: plt.ylabel('...')")
    if len(eje.collections) + len(eje.lines) + len(eje.patches) == 0:
        faltas.append("el eje esta vacio: el grafico no se dibujo sobre este eje")
    if escala_simetrica:
        norma = eje.collections[0].norm if len(eje.collections) else None
        if norma is None or norma.vmin != -1 or norma.vmax != 1:
            faltas.append("la escala de color no va de -1 a 1: revise center=0, vmin=-1 y vmax=1")
    if faltas:
        print(f"[{clave}] Todavia no esta completo:")
        for falta in faltas:
            print(f"[{clave}]   - {falta}")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
    else:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO: el grafico esta dibujado, titulado y etiquetado.")


def resumen_puntos_de_control():
    """Estado de las once tareas."""
    print("Punto de control")
    print("-" * 42)
    for clave in _CLAVES:
        estado = "correcto" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logrados = sum(1 for c in _CLAVES if _RESULTADOS.get(c))
    print("-" * 42)
    print(f"{logrados} de {len(_CLAVES)} {'correctas' if logrados != 1 else 'correcta'}.")


print("Verificador listo. Las tareas se comprueban con comprobar('T1', su_variable).")

---

## Paso 0.1 · Las columnas que importan

**El concepto.** 48 columnas son más de mil pares posibles. Si uno prueba todos, **va a encontrar**
correlaciones altas por puro azar: eso son las correlaciones espurias del bloque 1. La defensa es
elegir de antemano el conjunto de variables que tiene sentido mirar, y no salir a rastrillar.

Las dos listas de abajo ya están definidas y se usan en casi todas las tareas:

- `columnas_puntaje` — los **seis** puntajes, incluido `puntaje_global`.
- `columnas_modulo` — los **cinco** módulos, **sin** `puntaje_global`. Esta es la lista para buscar
  hallazgos de verdad.

Fíjese también en la fila `min` de la tabla que sale: ahí está la trampa del dataset.

In [ ]:
# Las columnas que sí vamos a usar. No hay que tocar las otras 42.
columnas_puntaje = [
    'puntaje_global',
    'punt_comp_ciud',
    'punt_comu_escr',
    'punt_ingles',
    'punt_lect_crit',
    'punt_razo_cuant'
]

# Sin puntaje_global: esta es la lista para buscar hallazgos de verdad
columnas_modulo = [c for c in columnas_puntaje if c != 'puntaje_global']

# El orden lógico de los estratos, para que los gráficos no salgan alfabéticos
orden_estratos = ['Estrato 1', 'Estrato 2', 'Estrato 3',
                  'Estrato 4', 'Estrato 5', 'Estrato 6']

df[columnas_puntaje].describe()

## Paso 0.2 · La limpieza de los -89

**El concepto.** Un **centinela** es un valor que ocupa el lugar de un dato faltante fingiendo ser un
dato normal. Aquí el centinela es -89. Una prueba estandarizada no da puntajes negativos, así que ese
número no mide nada: dice "no hay dato".

Es peor que una celda vacía. Una celda vacía la lee pandas como `NaN`, y `NaN` se propaga o se ignora
según la operación, pero **avisa**: `info()` lo muestra, `isna()` lo cuenta. El -89 no avisa. Entra en
la media, entra en la correlación, y aparece como un outlier fantasma en todos los boxplots.

| Columna | Registros con -89 |
|---------|-------------------|
| `punt_comu_escr` | 427 |
| `punt_ingles` | 29 |
| `punt_comp_ciud` | 19 |
| `punt_lect_crit` | 8 |
| `punt_razo_cuant` | 5 |

**Los comandos que usa la celda, explicados.**

```python
serie.mask(condicion)              # reemplaza por NaN donde la condicion es verdadera
df.dropna(subset=['a', 'b'])       # borra las filas con NaN en esas columnas
df.copy()                          # una copia independiente, para no dañar el original
```

`.mask()` es el negativo de un filtro: en vez de quedarse con las filas buenas, marca los valores
malos **dentro** de la columna. Después `dropna(subset=...)` elimina las filas incompletas.

**Por qué `df.copy()` y no trabajar sobre `df`.** Porque queremos poder comparar el antes y el
después, y porque el original en memoria es lo único que nos queda del archivo tal como venía.

Esta celda ya está escrita. Ejecútela y lea los tres números que imprime.

In [ ]:
print('Registros con el centinela -89, por columna:')
for col in columnas_puntaje:
    n = (df[col] < 0).sum()
    if n > 0:
        print(f'  {col:20s} {n:4,}')

# Paso 1: convertir el centinela en NaN de verdad
df_limpio = df.copy()
for col in columnas_puntaje:
    df_limpio[col] = df_limpio[col].mask(df_limpio[col] < 0)

# Paso 2: eliminar las filas incompletas
df_limpio = df_limpio.dropna(subset=columnas_puntaje)

# Paso 3: verificar. Si el mínimo sigue siendo negativo, la limpieza no sirvió.
print()
print(f'Filas antes:   {len(df):,}')
print(f'Filas después: {len(df_limpio):,}')
print(f'Eliminadas:    {len(df) - len(df_limpio):,}')
print()
print('Mínimo de cada puntaje después de limpiar:')
print(df_limpio[columnas_puntaje].min().to_string())

**Tu respuesta.** Explique con sus palabras por qué un centinela como -89 es más peligroso que una
celda vacía. Piense en qué pasa con la media y con la correlación si nadie lo limpia.

*Tu respuesta:*

## Paso 0.3 · Reconocimiento: mire los valores antes de agrupar

**El concepto.** El mismo reflejo de la clase 2, aplicado a categóricas: antes de filtrar o agrupar
por una columna de texto, hay que ver **cómo están escritos los valores**. `Estrato 1` con espacio no
es `Estrato1` sin espacio, y `Oficial` no es `OFICIAL`.

**Los comandos.**

```python
df['columna'].unique()          # los valores distintos, tal como estan escritos
df['columna'].value_counts()    # los mismos, con cuantas filas tiene cada uno
```

Esta celda está escrita. Ejecútela y **no cierre la salida**: de aquí va a copiar los nombres exactos.

In [ ]:
for col in ['estrato', 'tipo_col', 'sexo']:
    print(col, '->', df_limpio[col].unique().tolist())
print()
print(df_limpio['tipo_col'].value_counts().to_string())

---

# Parte A · Correlaciones entre puntajes

**Qué se practica aquí.** Los pasos 2 y 3 del marco bivariado sobre pares de variables **numéricas**:
graficar, cuantificar con r, y leer el número sin decir de más.

**El concepto, en tres frases.** El coeficiente de correlación de Pearson, **r**, resume en un solo
número qué tan bien se ajusta una recta a una nube de puntos. Va de -1 a +1: el **signo** dice la
dirección (si una sube, la otra sube o baja) y el **valor absoluto** dice la fuerza. No tiene
unidades y no es un porcentaje: r = 0,5 no es "la mitad de la relación".

| \|r\| | Lectura |
|-------|---------|
| 0,0 - 0,3 | Débil |
| 0,3 - 0,7 | Moderada |
| 0,7 - 1,0 | Fuerte |

**Y lo que r no ve:** las curvas (una relación en U perfecta da r cercano a 0), los outliers (un solo
punto extremo mueve el número) y los grupos. Por eso el paso 2 va antes que el paso 3.

### Tarea 1 · La matriz de los seis puntajes

**La pregunta.** ¿Cómo correlaciona cada puntaje con cada otro puntaje?

**El concepto.** Una **matriz de correlación** es el r de todos contra todos, en una tabla cuadrada.
Dos cosas que hay que saber leer: la **diagonal siempre vale 1** (toda variable correlaciona perfecto
consigo misma, y eso no es un hallazgo), y la matriz es **simétrica**, así que la mitad de arriba y la
de abajo dicen lo mismo.

**Los comandos.**

```python
df[lista_de_columnas].corr()      # la matriz de todos contra todos
matriz.round(2)                   # dos decimales, para poder leerla
```

**Lo que decide usted.** Sobre qué DataFrame se calcula (¿el original o el limpio?) y cuál de las dos
listas de columnas entra. Elegir mal aquí no produce ningún error: produce números equivocados.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Calcule la matriz de correlación de los SEIS puntajes, sobre el DataFrame limpio.
# 2. Guárdela en matriz_puntajes.
# 3. Muéstrela redondeada a 2 decimales.

matriz_puntajes = None

In [ ]:
comprobar('T1', matriz_puntajes, decimales=2)

### Tarea 2 · El heatmap de esa matriz

**La pregunta.** La misma información de la tarea 1, pero legible de un vistazo.

**El concepto.** Un **mapa de calor** (*heatmap*) pinta cada celda de la matriz con un color según su
valor. Sirve para leer por bloques en vez de celda por celda. El parámetro que decide si el gráfico
dice la verdad es `center=0`: pone el color neutro en el cero, de modo que el rojo signifique
positivo y el azul negativo. **Sin `center=0`, la escala de color se reparte entre el mínimo y el
máximo de la matriz, y un r de 0,1 puede verse tan intenso como uno de 0,9.** Es un gráfico que miente
sin dar ningún error.

`vmin=-1, vmax=1` fija la escala al rango real de r, para que dos heatmaps distintos se puedan
comparar entre sí.

**Los comandos.**

```python
plt.figure(figsize=(9, 7))
eje = sns.heatmap(matriz,
                  annot=True,        # escribe el numero dentro de cada celda
                  cmap='coolwarm',   # azul negativo, rojo positivo
                  center=0,          # el color neutro en cero
                  vmin=-1, vmax=1,   # la escala fija
                  fmt='.2f',
                  square=True,
                  linewidths=0.5)
plt.title('...')
plt.tight_layout()
plt.show()
```

**Lo que decide usted.** Qué matriz entra, y el título. Guarde el resultado de `sns.heatmap(...)` en
la variable `eje_heatmap`: seaborn devuelve el eje sobre el que dibujó, y es lo que revisa la
comprobación.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Abra una figura del tamaño que le sirva.
# 2. Dibuje el heatmap de matriz_puntajes y guarde el eje en eje_heatmap.
# 3. Póngale título.
# 4. plt.tight_layout() y plt.show()

eje_heatmap = None

In [ ]:
comprobar_grafico('T2', eje_heatmap, con_ejes=False, escala_simetrica=True)

**Tu respuesta.** Mire el heatmap por bloques. ¿Qué fila o columna se ve claramente más pálida que
las demás? ¿Y qué fila se ve intensa contra todo, y por qué eso **no** es un hallazgo?

*Tu respuesta:*

### Tarea 3 · Las tres correlaciones más fuertes

**La pregunta.** Excluyendo la diagonal y excluyendo `puntaje_global`, ¿cuáles son los tres pares de
puntajes más correlacionados?

**El concepto.** Dos exclusiones, y las dos son conceptuales, no de código.

- **La diagonal** vale 1 siempre. Reportarla es reportar que una variable se parece a sí misma.
- **`puntaje_global`** es un **compuesto**: se calcula a partir de los otros cinco. Correlaciona alto
  con todos **por construcción**. Reportar "descubrí que el puntaje global correlaciona 0,75 con
  lectura crítica" no es descubrir nada: es descubrir cómo se calcula el puntaje global. Es el error
  número uno que produce este dataset.

La función `top_correlaciones()` de la celda de abajo ya está escrita: recorre solo el triángulo
superior de la matriz (para no contar cada par dos veces ni tropezar con la diagonal) y devuelve los
pares ordenados por fuerza.

**Los comandos.**

```python
matriz = df[lista_de_columnas].corr()
top = top_correlaciones(matriz, n=3)      # lista de (columna_a, columna_b, r)
for a, b, r in top:
    print(a, b, round(r, 3))
```

**Lo que decide usted.** **Cuál de las dos listas de columnas** entra en la matriz de este punto. Ahí
está toda la tarea.

In [ ]:
def top_correlaciones(matriz, n=3):
    """
    Extrae los n pares con mayor correlación absoluta, sin la diagonal
    y sin repetir pares.
    """
    pares = []
    columnas = matriz.columns
    for i in range(len(columnas)):
        for j in range(i + 1, len(columnas)):   # solo el triángulo superior
            pares.append((columnas[i], columnas[j], matriz.iloc[i, j]))

    pares.sort(key=lambda t: abs(t[2]), reverse=True)
    return pares[:n]


print('Función definida.')

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Calcule la matriz de correlación de los CINCO módulos y guárdela en matriz_modulos.
# 2. Pásela a top_correlaciones(..., n=3) y guarde el resultado en top3.
# 3. Imprima cada par con su r, y diga si la fuerza es débil, moderada o fuerte.

matriz_modulos = None
top3 = None

In [ ]:
# La normalización se hace aquí para que el resultado no dependa del orden en que
# usted haya escrito la lista de columnas.
if top3 is None:
    comprobar('T3', None)
else:
    comprobar('T3', [(min(a, b), max(a, b), round(float(r), 3)) for a, b, r in top3])

### Explique las tres

Para cada par: qué mide cada variable, si la relación tiene sentido, y qué **confusora** podría estar
detrás. Recuerde el paso 4 del marco: una correlación sin mecanismo no se publica.

**Correlación 1** — par, r, ¿tiene sentido lógico?, ¿confusora posible?

*Tu respuesta:*

**Correlación 2** — par, r, ¿tiene sentido lógico?, ¿confusora posible?

*Tu respuesta:*

**Correlación 3** — par, r, ¿tiene sentido lógico?, ¿confusora posible?

*Tu respuesta:*

### El módulo raro

Mire la fila de `punt_comu_escr` en el heatmap. Sus correlaciones con todo lo demás son notoriamente
más bajas que las de los otros módulos entre sí.

**Tu respuesta.** ¿Por qué comunicación escrita se comportaría distinto de los demás módulos? Pista:
piense en **cómo se califica** una prueba de escritura frente a una de selección múltiple, y quién la
califica.

*Tu respuesta:*

### Tarea 4 · El scatter del par más fuerte

**La pregunta.** ¿Cómo se ve, dibujada, la relación del par número 1 de su top 3?

**El concepto.** El paso 2 del marco va **antes** que el paso 3, y aquí lo hacemos al revés a
propósito para que vea por qué. El número ya lo tiene; ahora mire la nube y compruebe que el número no
le estaba escondiendo nada: ¿hay una curva? ¿hay puntos sueltos muy lejos? ¿hay dos grupos separados?

Con 24.897 filas, un scatter sin transparencia es una mancha. `alpha` es la transparencia de cada
punto: donde hay muchos encimados se ve oscuro, donde hay pocos se ve claro. El gráfico pasa a mostrar
**densidad**, no solo posición. Y `.sample()` toma una muestra aleatoria; `random_state` fija el azar
para que a todos les salga la misma.

**Los comandos.**

```python
muestra = df.sample(3000, random_state=42)
fig, eje = plt.subplots(figsize=(8, 6))
eje.scatter(muestra['columna_x'], muestra['columna_y'], alpha=0.2, s=12)
eje.set_xlabel('...')
eje.set_ylabel('...')
eje.set_title('...')
plt.tight_layout()
plt.show()
```

**Lo que decide usted.** Cuáles son las dos columnas del par más fuerte, y qué dice el título. Guarde
el eje en `eje_scatter`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Tome una muestra de 3.000 filas de df_limpio, con random_state=42.
# 2. Dibuje el scatter de las dos columnas del par más fuerte, con alpha bajo.
# 3. Guarde el eje en eje_scatter, y póngale título y etiquetas a los dos ejes.
#    Buen título: el nombre del par y el valor de r.

eje_scatter = None

In [ ]:
comprobar_grafico('T4', eje_scatter)

**Tu respuesta.** ¿La nube confirma el número, o el número le estaba escondiendo algo? Diga
explícitamente si ve una curva, outliers o grupos separados.

*Tu respuesta:*

---

# Parte B · Categórica contra numérica

**Qué se practica aquí.** El otro caso de la tabla de elección de gráfico: cuando una de las dos
variables es **categórica**, el scatter no sirve porque no hay eje numérico donde poner las
categorías.

| Variable 1 | Variable 2 | Gráfico | Métrica |
|------------|------------|---------|---------|
| Numérica | Numérica | Diagrama de dispersión | Coeficiente r |
| Categórica | Numérica | Boxplot, o barras de medias | Comparación de medias/medianas |
| Categórica | Categórica | Tabla cruzada, mapa de calor | Conteos y proporciones |

**Boxplot o barras de medias**, que es la duda que siempre sale:

- **Boxplot** cuando importa la **dispersión**: muestra mediana, cuartiles, bigotes y outliers. Es el
  que casi siempre conviene.
- **Barras de medias** cuando importa solo comparar niveles y la audiencia no lee boxplots. Esconde
  la dispersión, así que se acompaña de una nota.

**Y una advertencia que vale para toda esta parte:** comparar medias entre grupos no dice nada sobre
causas. Un gradiente limpio por estrato es una **asociación**, y las asociaciones tienen siempre las
mismas tres explicaciones posibles.

### Tarea 5 · Puntaje global por estrato

**La pregunta.** ¿Cómo se distribuye el puntaje global en cada estrato socioeconómico?

**El concepto.** Categórica (`estrato`) contra numérica (`puntaje_global`): boxplot. Un detalle que
cambia la lectura del gráfico: seaborn ordena las categorías como se le antoje si no se le dice, y un
estrato tiene **orden natural**. El parámetro `order` fuerza el orden lógico, y sin él la lectura
"sube con el estrato" no se puede hacer. `orden_estratos` ya está definido más arriba.

Note que `orden_estratos` deja fuera la categoría `ND/NE`, que no es un estrato sino la ausencia del
dato. Excluirla del gráfico es una decisión defendible; esconderla sin decirlo, no. Por eso lo
decimos aquí.

**Los comandos.**

```python
plt.figure(figsize=(11, 6))
eje = sns.boxplot(x='columna_categorica', y='columna_numerica',
                  data=df, order=lista_de_orden)
plt.xlabel('...')
plt.ylabel('...')
plt.title('...')
plt.tight_layout()
plt.show()
```

**Lo que decide usted.** Cuál de las dos variables va en `x` y cuál en `y` (la categórica va en el eje
horizontal), y los textos. Guarde el eje en `eje_box_estrato`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Boxplot de puntaje_global por estrato, sobre df_limpio, con order=orden_estratos.
# 2. Guarde el eje en eje_box_estrato.
# 3. Título y etiquetas en los dos ejes.

eje_box_estrato = None

In [ ]:
comprobar_grafico('T5', eje_box_estrato)

### Tarea 6 · Puntaje global por tipo de colegio

**La pregunta.** ¿Cómo se distribuye el puntaje global según el tipo de colegio de origen?

**El concepto.** El mismo gráfico, sobre otra categórica. Aquí **no hay orden natural** que imponer
—Oficial y Privado no van uno antes que el otro—, así que `order` sobra. Es la diferencia entre una
categórica **ordinal** (estrato: hay un orden) y una **nominal** (tipo de colegio: no lo hay), y
decide si el parámetro `order` es obligatorio o decorativo.

**Los comandos.** Los mismos de la tarea 5, sin `order`.

**Lo que decide usted.** La columna categórica y los textos. Guarde el eje en `eje_box_tipo`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Boxplot de puntaje_global por tipo de colegio, sobre df_limpio.
# 2. Guarde el eje en eje_box_tipo.
# 3. Título y etiquetas en los dos ejes.

eje_box_tipo = None

In [ ]:
comprobar_grafico('T6', eje_box_tipo)

### Tarea 7 · Las medias por estrato, con su conteo

**La pregunta.** ¿Cuál es el puntaje global promedio de cada estrato, y de cuánta gente estamos
hablando en cada uno?

**El concepto.** El boxplot muestra la forma; el número la fija. Y **una media sin su conteo no se
puede interpretar**: una media de 189 sobre 175 personas y una media de 189 sobre 9.000 personas son
afirmaciones muy distintas, aunque el número sea el mismo. Por eso se piden las dos cosas de un golpe
con `.agg(['mean', 'count'])`.

**Los comandos.**

```python
df.groupby('columna_categorica')['columna_numerica'].agg(['mean', 'count'])
tabla.round(1)
tabla.sort_values('mean', ascending=False)
```

**Lo que decide usted.** La columna por la que se agrupa, la que se resume, y si le sirve más
ordenada por media o por el orden de los estratos.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Agrupe df_limpio por estrato y calcule media y conteo de puntaje_global.
# 2. Guárdelo en medias_estrato y muéstrelo redondeado a 1 decimal.

medias_estrato = None

In [ ]:
comprobar('T7', medias_estrato, decimales=1)

### Interpretación de la parte B

**Pregunta 1.** ¿Qué patrón ve en el puntaje por estrato? Descríbalo en una frase, sin usar ningún
verbo causal.

*Tu respuesta:*

**Pregunta 2.** ¿Qué **no** se puede concluir de ese patrón? Sea explícito y **nombre al menos dos
variables confusoras plausibles**: cosas que van juntas con el estrato y que sí podrían explicar el
puntaje.

*Tu respuesta:*

**Pregunta 3.** Mire la columna `count` de la tarea 7. ¿Hay algún grupo tan pequeño como para que su
media no sea confiable? ¿Cuál, y a partir de qué tamaño empezaría a desconfiar?

*Tu respuesta:*

---

# Parte C · Chequeo de paradoja de Simpson

**Es la parte que más pesa, y es la única donde no se le da el orden de los comandos.**

**El concepto.** La **paradoja de Simpson**: una tendencia que aparece en el conjunto completo puede
**desaparecer o invertirse** cuando se parte por grupos. El caso canónico son las admisiones de
posgrado de Berkeley en 1973: globalmente admitían al 44% de los hombres y al 35% de las mujeres, lo
que parecía discriminación evidente; al separar por departamento, la mayoría de los departamentos
admitía a las mujeres en igual o mayor proporción. La confusora era el departamento al que se
postulaba.

**Confusora y paradoja no son lo mismo.** La confusora es la **causa**: una tercera variable que está
detrás de las dos. La paradoja de Simpson es el **síntoma**: lo que le pasa a los números cuando esa
confusora está desbalanceada entre los grupos.

**El chequeo, en cuatro pasos:**

1. Calcule la relación en el conjunto completo.
2. Recalcúlela **dentro de cada grupo**.
3. Compare. Si el signo se invierte o la fuerza cambia mucho, hay una confusora estructural.
4. Reporte lo que encontró, **incluso si es "no hay paradoja"**.

**Lo que se chequea hoy:** los colegios privados tienen media más alta que los oficiales. ¿Esa ventaja
sobrevive al partir por estrato, o era solo un efecto de composición?

> **El entregable es el chequeo, no la paradoja.** Si la relación se sostiene, ese es un resultado
> completo y correcto: que una relación resista la partición la hace **más** creíble, no menos.
> Forzar los datos hasta sacar un titular es exactamente lo contrario de lo que esta clase enseña.

### El inventario de comandos

Estos son todos los comandos que necesitan las cuatro tareas de esta parte. Todos los ha usado ya, o
están explicados aquí. **Lo que no se le da es el orden en que se arman**, y eso es deliberado: en las
sustentaciones de los momentos 1, 2 y 3 nadie le va a dar la secuencia.

```python
df.groupby('columna')['numerica'].agg(['mean', 'count'])
pd.pivot_table(df, index='filas', columns='columnas', values='numerica', aggfunc='mean')
pd.pivot_table(df, index='filas', columns='columnas', values='numerica', aggfunc='size')
tabla.loc[lista_de_filas, lista_de_columnas]
tabla['nueva'] = tabla['columna_a'] - tabla['columna_b']
tabla.sort_values('columna', ascending=False)
tabla.round(1)
tabla.copy()
```

**`pd.pivot_table()` es la herramienta nueva.** Cruza dos categóricas y calcula un resumen de una
numérica en cada celda del cruce: es un GroupBy de dos dimensiones presentado como tabla. `index` son
las filas, `columns` son las columnas, `values` es lo que se resume y `aggfunc` es cómo se resume.
Con `aggfunc='size'` no resume nada: cuenta cuántas filas cayeron en cada celda.

### Tarea 8 · El resultado global

**La pregunta.** ¿Cuál es el puntaje global promedio de cada tipo de colegio, y cuánta gente hay en
cada uno? Este es el paso 1 del chequeo: la relación en el conjunto completo.

**El concepto.** Es la afirmación que vamos a poner a prueba. Antes de partir nada hay que dejar el
resultado global escrito, porque si no, no hay contra qué comparar.

**Lo que decide usted.** Qué comando del inventario sirve aquí y sobre qué columna. Deje las cuatro
categorías tal como salen: quitar `Otros` y `Sin datos` cambia lo que se está comparando, y esa
decisión se toma explícitamente, no de pasada.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Media y conteo de puntaje_global por tipo de colegio, sobre df_limpio.
# 2. Guárdelo en medias_tipo_col y muéstrelo ordenado de mayor a menor media.

medias_tipo_col = None

In [ ]:
comprobar('T8', medias_tipo_col, decimales=1)

### Tarea 9 · El resultado partido por estrato

**La pregunta.** Dentro de cada estrato por separado, ¿cuál es el puntaje promedio de cada tipo de
colegio?

**El concepto.** Este es el paso 2 del chequeo. La partición se hace por la variable que uno sospecha
que es la confusora: aquí, el estrato. La pregunta que responde la tabla es si la ventaja del privado
existe **dentro** de cada estrato, o si aparecía en el global solo porque los privados están
concentrados en los estratos altos.

**Lo que decide usted.** Cuál de los tres comandos de agrupación del inventario produce una tabla de
dos entradas, qué va en `index`, qué va en `columns` y qué en `values`. La tabla completa, sin filtrar
todavía.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Cruce estrato (filas) con tipo de colegio (columnas) y resuma la media de puntaje_global.
# 2. Guárdelo en tabla_medias y muéstrelo redondeado a 1 decimal.

tabla_medias = None

In [ ]:
comprobar('T9', tabla_medias, decimales=1)

### Tarea 10 · Los tamaños de cada celda

**La pregunta.** ¿Cuántos estudiantes hay en cada celda de ese cruce?

**El concepto.** Sin esta tabla, la anterior es intrepretable a medias. Una media calculada sobre
cinco personas y una calculada sobre cinco mil se ven idénticas en pantalla y no valen lo mismo. Y hay
una segunda razón, más importante hoy: **el desbalance de los tamaños es lo que produce las paradojas
de Simpson**. Si los oficiales están concentrados en los estratos bajos y los privados en los altos,
el promedio global de cada tipo está midiendo, en parte, la composición por estrato.

**Lo que decide usted.** Qué cambia respecto a la tarea anterior. Es un solo parámetro.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. La misma tabla cruzada, pero contando estudiantes en vez de promediar puntajes.
# 2. Guárdela en tabla_tamanos y muéstrela.

tabla_tamanos = None

In [ ]:
comprobar('T10', tabla_tamanos, decimales=1)

### Tarea 11 · La comparación, estrato por estrato

**La pregunta.** En cada uno de los seis estratos, ¿cuánta ventaja le saca el colegio privado al
oficial?

**El concepto.** Este es el paso 3 del chequeo: comparar. Una tabla de dos columnas y una resta
responde la pregunta de un vistazo, y deja la conclusión en evidencia sin que haya que buscarla entre
celdas.

Dos decisiones que importan y que no se le dan:

- **Qué filas y qué columnas se conservan.** La tabla de la tarea 9 trae cuatro tipos de colegio y una
  fila `ND/NE`. La comparación es entre dos tipos, en los seis estratos con nombre.
- **El orden de la resta.** "La ventaja del privado" tiene un signo, y si lo invierte, la tabla dice
  lo contrario de lo que usted va a escribir debajo.

**Lo que decide usted.** El ensamblaje completo, con los comandos del inventario. Si se atasca, parta
el problema: primero la tabla recortada, mírela, y solo después agregue la columna de la resta.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Parta de tabla_medias y quédese con los seis estratos y las dos columnas que se comparan.
#    Use .copy() para trabajar sobre una tabla propia.
# 2. Agregue una columna llamada 'diferencia' con la ventaja del privado sobre el oficial.
# 3. Guarde el resultado en comparacion y muéstrelo.

comparacion = None

In [ ]:
comprobar('T11', comparacion, decimales=1)

### Su conclusión del chequeo

**Pregunta 1.** ¿La ventaja de Privado sobre Oficial se mantiene dentro de cada estrato, se invierte,
o desaparece? Responda mirando la columna `diferencia`, no de memoria.

*Tu respuesta:*

**Pregunta 2.** ¿Hay paradoja de Simpson aquí? Responda con honestidad y diga en qué se basa. Un "no"
argumentado es una respuesta completa.

*Tu respuesta:*

**Pregunta 3.** Mire la tabla de tamaños de la tarea 10. ¿Hay alguna celda tan pequeña que no permita
concluir nada? ¿Cuál, y qué haría usted con ella al escribir el informe?

*Tu respuesta:*

**Pregunta 4.** Los tamaños de grupo están muy desbalanceados: hay muchos oficiales en los estratos
bajos y muchos privados en los altos. Ese desbalance es justamente el que **podría** haber producido
una paradoja. Explique por qué no la produjo en este caso.

*Tu respuesta:*

---

## Punto de control

Ejecute la celda de abajo para ver cuántas de las once tareas quedaron correctas.

Si alguna sigue pendiente, no pase de largo. Si está en el salón, levante la mano ahora, que el
profesor está aquí para eso.

In [ ]:
resumen_puntos_de_control()

---

# Parte D · Reflexión

Responda en español, dos o tres frases por pregunta. Estas no se comprueban con código: son las que se
leen en la dimensión **Ser**.

**1. Causalidad.** Un titular de prensa dice: *"Estudiar en colegio privado mejora el puntaje del
Saber Pro"*. A partir de su análisis, ¿ese titular es defendible? Reescríbalo de forma que sí lo sea.

*Tu respuesta:*

**2. Correlaciones espurias.** Este dataset tiene 48 columnas: más de mil pares posibles. Si usted los
probara todos, encontraría correlaciones altas sin ningún sentido. ¿Cómo se protege de reportar una
correlación espuria?

*Tu respuesta:*

**3. Del reto al Momento 1.** ¿Qué análisis bivariado va a incluir en la entrega del Momento 1 con el
dataset de su equipo? Escriba el par de variables, el gráfico que le corresponde según la tabla de
tipos, y por qué ese par y no otro.

*Tu respuesta:*

---

## Opcional · Solo si terminó todo

No se comprueban y no entran en la retroalimentación.

**A. Un segundo chequeo de Simpson.** La media de `puntaje_global` por `sexo`, global y partida por
`areac_snies` (área de conocimiento). ¿Se invierte en alguna área? Es el mismo procedimiento de la
parte C con otras dos categóricas.

**B. Pair plot.** `sns.pairplot(df_limpio.sample(500, random_state=42), vars=columnas_modulo)`. Es la
matriz de correlación convertida en gráficos: donde la matriz da un número, el pair plot da la forma.

**C. Más allá de los puntajes.** Correlación de los puntajes contra `edad` y contra `pbm` (puntaje
básico de matrícula, un indicador socioeconómico). ¿Qué encuentra, y qué mecanismo podría explicarlo?

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---

## Antes de entregar

1. **Kernel → Restart and Run All.** Si algo revienta, arréglelo. Un cuaderno que no corre de arriba
   a abajo le pone techo a la dimensión Hacer.
2. Ejecute el punto de control y verifique que las once tareas están correctas.
3. Verifique que **todas** las celdas *Tu respuesta:* están escritas. El número no es el análisis.
4. Revise que ningún texto suyo diga que una variable **causa** otra.
5. Guarde como `reto_clase05_APELLIDO.ipynb` y súbalo antes del inicio de la clase 6.